# Analysis of OBI vs Mid-Price Movements

This notebook connects to the Delta Lake storage populated by the `RealTimeEngine` and visualizes the relationship between Order Book Imbalance (OBI) and Mid-Price changes.

In [ ]:
import polars as pl
import matplotlib.pyplot as plt
from deltalake import DeltaTable

# Load Delta Table
dt = DeltaTable('../storage/delta_lake')
df = pl.scan_pyarrow_dataset(dt.to_pyarrow_dataset())

# Preview data
df.head(5).collect()

In [ ]:
# Select a specific symbol for visualization
symbol = 'TICK1'
sym_df = df.filter(pl.col('symbol') == symbol).collect().sort('timestamp_ns')

# Calculate mid-price and future mid-price changes
sym_df = sym_df.with_columns([
    ((pl.col('bid_price') + pl.col('ask_price')) / 2).alias('mid_price')
])

sym_df = sym_df.with_columns([
    (pl.col('mid_price').shift(-1) - pl.col('mid_price')).alias('mid_price_change')
])

In [ ]:
# Plotting OBI vs Mid-price 
fig, ax1 = plt.subplots(figsize=(10, 5))

ax1.plot(sym_df['timestamp_ns'], sym_df['mid_price'], color='b', label='Mid Price')
ax1.set_xlabel('Timestamp (ns)')
ax1.set_ylabel('Mid Price', color='b')

ax2 = ax1.twinx()
ax2.plot(sym_df['timestamp_ns'], sym_df['obi_avg'], color='r', alpha=0.5, label='OBI')
ax2.set_ylabel('Order Book Imbalance', color='r')

plt.title(f'Mid Price and OBI over Time ({symbol})')
plt.show()